# Lab | BabyAGI with agent

**Change the planner objective below by changing the objective and the associated prompts and potential tools and agents - Wear your creativity and AI engineering hats
You can't get this wrong!**

You would need the OpenAI API KEY and the [SerpAPI KEY](https://serpapi.com/manage-api-keyhttps://serpapi.com/manage-api-key) to run this lab.


## BabyAGI with Tools

This notebook builds on top of [baby agi](baby_agi.html), but shows how you can swap out the execution chain. The previous execution chain was just an LLM which made stuff up. By swapping it out with an agent that has access to tools, we can hopefully get real reliable information

## Install and Import Required Modules

In [ ]:
!pip install langchain_community langchain_experimental langchain_openai

: 

In [2]:
print("START")

from langchain_experimental.autonomous_agents import BabyAGI

print("IMPORTED")

START
IMPORTED


In [3]:
import sys
print(sys.executable)

c:\Users\profe\Documents\Ironhack\lab-babyagi-with-agent\venv_clean\Scripts\python.exe


In [4]:
import pkgutil
print("langchain_experimental" in [m.name for m in pkgutil.iter_modules()])

True


In [5]:
try:
    from langchain_experimental.autonomous_agents import BabyAGI
    print("SUCCESS")
except Exception as e:
    print("ERROR:", e)

SUCCESS


In [6]:
from langchain_experimental.autonomous_agents import BabyAGI
print("BabyAGI works ✅")

BabyAGI works ✅


In [7]:
# =========================================================
# 1. Standard library
# =========================================================
from typing import Optional, List

# =========================================================
# 2. Environment
# =========================================================
from dotenv import load_dotenv
load_dotenv()

# =========================================================
# 3. Core LangChain
# =========================================================
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate

# =========================================================
# 4. Experimental (BabyAGI)
# =========================================================
from langchain_experimental.autonomous_agents import BabyAGI

# =========================================================
# 5. Models + Embeddings (UPDATED)
# =========================================================
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# =========================================================
# 6. Vector Store (memory)
# =========================================================
from langchain_community.vectorstores import FAISS

## Connect to the Vector Store

Depending on what vectorstore you use, this step may look different.

In [8]:
# # %pip install faiss-cpu > /dev/null
# # %pip install google-search-results > /dev/null
# from langchain.docstore import InMemoryDocstore
# from langchain_community.vectorstores import FAISS

# =========================================================
# Vector Store Setup (FAISS + In-Memory Docstore)
# =========================================================

# NOTE:
# Install dependencies in terminal (not inside notebook):
# pip install faiss-cpu google-search-results

from langchain.docstore import InMemoryDocstore
from langchain_community.vectorstores import FAISS

In [9]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
SERPAPI_API_KEY = os.getenv('SERPAPI_API_KEY')

In [10]:
# Define your embedding model
embeddings_model = OpenAIEmbeddings()
# Initialize the vectorstore as empty
import faiss

embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(embeddings_model.embed_query, index, InMemoryDocstore({}), {})

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


## Define the Chains

BabyAGI relies on three LLM chains:
- Task creation chain to select new tasks to add to the list
- Task prioritization chain to re-prioritize tasks
- Execution Chain to execute the tasks


NOTE: in this notebook, the Execution chain will now be an agent.

In [13]:
# from langchain.agents import AgentExecutor, Tool, ZeroShotAgent
# from langchain.chains import LLMChain
# from langchain_community.utilities import SerpAPIWrapper
# from langchain_openai import OpenAI

# todo_prompt = PromptTemplate.from_template(
#     "You are a planner who is an expert at coming up with a todo list for a given objective. Come up with a todo list for this objective: {objective}"
# )
# todo_chain = LLMChain(llm=OpenAI(temperature=0), prompt=todo_prompt)
# search = SerpAPIWrapper()
# tools = [
#     Tool(
#         name="Search",
#         func=search.run,
#         description="useful for when you need to answer questions about current events",
#     ),
#     Tool(
#         name="TODO",
#         func=todo_chain.run,
#         description="useful for when you need to come up with todo lists. Input: an objective to create a todo list for. Output: a todo list for that objective. Please be very clear what the objective is!",
#     ),
# ]


# prefix = """You are an AI who performs one task based on the following objective: {objective}. Take into account these previously completed tasks: {context}."""
# suffix = """Question: {task}
# {agent_scratchpad}"""
# prompt = ZeroShotAgent.create_prompt(
#     tools,
#     prefix=prefix,
#     suffix=suffix,
#     input_variables=["objective", "task", "context", "agent_scratchpad"],
# )

# =========================================================
# 1. Imports
# =========================================================
from langchain.agents import Tool, AgentExecutor, create_openai_tools_agent
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate, ChatPromptTemplate

from langchain_community.utilities import SerpAPIWrapper
from langchain_openai import ChatOpenAI


# =========================================================
# 2. LLM (modern)
# =========================================================
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)


# =========================================================
# 3. TODO Chain (Task Creation Helper)
# =========================================================
todo_prompt = PromptTemplate.from_template(
    "You are a planner who is an expert at creating a todo list for a given objective.\n"
    "Objective: {objective}\n\n"
    "Return a clear, step-by-step todo list."
)

todo_chain = LLMChain(
    llm=llm,
    prompt=todo_prompt
)


# =========================================================
# 4. Tools
# =========================================================
search = SerpAPIWrapper()

tools = [
    Tool(
        name="search",
        func=search.run,
        description="Use this for current events or external information."
    ),
    Tool(
        name="todo_planner",
        func=lambda q: todo_chain.invoke({"objective": q})["text"],
        description=(
            "Use this to generate a structured todo list for a given objective. "
            "Input should be a clear objective."
        ),
    ),
]


# =========================================================
# 5. Agent Prompt (modern)
# =========================================================
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an AI that executes one task at a time based on an objective.\n"
     "Take into account previously completed tasks (context) when reasoning."),
    
    ("human",
     "Objective: {objective}\n"
     "Previous tasks: {context}\n"
     "Current task: {task}"),
    
    ("placeholder", "{agent_scratchpad}")
])


# =========================================================
# 6. Create Agent (modern tool-calling)
# =========================================================
agent = create_openai_tools_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [15]:
# llm = OpenAI(temperature=0)
# llm_chain = LLMChain(llm=llm, prompt=prompt)
# tool_names = [tool.name for tool in tools]
# agent = ZeroShotAgent(llm_chain=llm_chain, allowed_tools=tool_names)
# agent_executor = AgentExecutor.from_agent_and_tools(
#     agent=agent, tools=tools, verbose=True
# )

# =========================================================
# Modern Agent Setup (replaces ZeroShotAgent)
# =========================================================
from langchain.agents import create_openai_tools_agent, AgentExecutor
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI


# 1. LLM (modern)
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)


# 2. Prompt (structured for agent)
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an AI that executes a single task based on an objective.\n"
     "Use tools when helpful. Be concise and accurate."),
    
    ("human",
     "Objective: {objective}\n"
     "Previous tasks: {context}\n"
     "Current task: {task}"),
    
    ("placeholder", "{agent_scratchpad}")
])


# 3. Create agent (modern tool-calling)
agent = create_openai_tools_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)


# 4. Executor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)


### Run the BabyAGI

Now it's time to create the BabyAGI controller and watch it try to accomplish your objective.

In [16]:
OBJECTIVE = "Write a weather report for SF today"

In [17]:
# Logging of LLMChains
verbose = False
# If None, will keep on going forever
max_iterations: Optional[int] = 3
baby_agi = BabyAGI.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    task_execution_chain=agent_executor,
    verbose=verbose,
    max_iterations=max_iterations,
)

In [20]:
# baby_agi({"objective": OBJECTIVE})

# Modernized
from langchain_core.runnables import RunnableLambda

execution_chain = RunnableLambda(
    lambda inputs: agent_executor.invoke(inputs)["output"]
)